# Notebook 03: Add a New Problem Scaffold (DCC26)

Reference implementation of a minimal, reproducible benchmark problem scaffold.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

Use this as a pattern for structuring new benchmark problems with explicit contracts and validation checks.


## What makes a new problem benchmark-ready

Benchmark value comes from clarity and comparability, not only simulator sophistication.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force install outside Colab


def pip_install(packages: list[str]):
    cmd = [sys.executable, '-m', 'pip', 'install', *packages]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)
BASE_PACKAGES = ['engibench[beams2d]', 'matplotlib', 'gymnasium', 'pybullet']
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    pip_install(BASE_PACKAGES)
    pip_install([ENGIOPT_GIT])

    try:
        import torch  # noqa: F401
    except Exception:
        pip_install(['torch', 'torchvision'])

    print('Dependency install complete.')
else:
    print('Skipping install (using current environment). Set FORCE_INSTALL=True to install here.')


### Step 1 - Import scaffold dependencies

Keep imports minimal and interface-focused.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem

import pybullet as p


### Step 2 - Implement PyBullet manipulator co-design problem contract

Ensure methods are deterministic and constraints/objectives are semantically explicit.


In [ ]:
class PlanarManipulatorCoDesignProblem(Problem[np.ndarray]):
    """Robotics co-design scaffold using a real PyBullet rollout loop."""

    version = 0
    objectives = (
        ("final_tracking_error_m", ObjectiveDirection.MINIMIZE),
        ("actuation_energy_j", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        target_x: Annotated[float, bounded(lower=0.20, upper=1.35)] = 0.85
        target_y: Annotated[float, bounded(lower=0.05, upper=1.20)] = 0.45
        payload_kg: Annotated[float, bounded(lower=0.0, upper=2.0)] = 0.8
        disturbance_scale: Annotated[float, bounded(lower=0.0, upper=0.30)] = 0.05

    @dataclass
    class Config(Conditions):
        sim_steps: Annotated[int, bounded(lower=60, upper=1200)] = 240
        dt: Annotated[float, bounded(lower=1e-4, upper=0.05)] = 1.0 / 120.0
        torque_limit: Annotated[float, bounded(lower=1.0, upper=50.0)] = 12.0
        max_iter: Annotated[int, bounded(lower=1, upper=300)] = 60

    dataset_id = "IDEALLab/planar_manipulator_codesign_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            target_x=self.config.target_x,
            target_y=self.config.target_y,
            payload_kg=self.config.payload_kg,
            disturbance_scale=self.config.disturbance_scale,
        )

        # Design vector = [link1_m, link2_m, motor_strength, kp, kd, damping]
        self.design_space = spaces.Box(
            low=np.array([0.25, 0.20, 2.0, 5.0, 0.2, 0.0], dtype=np.float32),
            high=np.array([1.00, 0.95, 30.0, 120.0, 18.0, 1.5], dtype=np.float32),
            dtype=np.float32,
        )

        @constraint
        def reachable_workspace(design: np.ndarray, target_x: float, target_y: float, **_) -> None:
            l1, l2 = float(design[0]), float(design[1])
            r = float(np.sqrt(target_x**2 + target_y**2))
            assert l1 + l2 >= r + 0.03, f"target radius {r:.3f} exceeds reach {l1+l2:.3f}"

        @constraint
        def gain_consistency(design: np.ndarray, **_) -> None:
            kp, kd = float(design[3]), float(design[4])
            assert kd <= 2.2 * np.sqrt(max(kp, 1e-6)), f"kd={kd:.3f} too high for kp={kp:.3f}"

        self.design_constraints = [reachable_workspace, gain_consistency]

    def _build_robot(self, l1: float, l2: float, payload_kg: float, damping: float) -> tuple[int, int]:
        p.resetSimulation()
        p.setGravity(0, 0, -9.81)

        link_masses = [0.5 + 0.2 * payload_kg, 0.35 + 0.25 * payload_kg]
        link_collision = [-1, -1]
        link_visual = [
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.025, length=l1, rgbaColor=[0.2, 0.5, 0.9, 1.0]),
            p.createVisualShape(p.GEOM_CAPSULE, radius=0.020, length=l2, rgbaColor=[0.9, 0.4, 0.2, 1.0]),
        ]
        qx = p.getQuaternionFromEuler([0.0, np.pi / 2.0, 0.0])

        robot = p.createMultiBody(
            baseMass=0.0,
            baseCollisionShapeIndex=-1,
            baseVisualShapeIndex=-1,
            basePosition=[0, 0, 0],
            linkMasses=link_masses,
            linkCollisionShapeIndices=link_collision,
            linkVisualShapeIndices=link_visual,
            linkPositions=[[0, 0, 0], [l1, 0, 0]],
            linkOrientations=[qx, qx],
            linkInertialFramePositions=[[l1 / 2.0, 0, 0], [l2 / 2.0, 0, 0]],
            linkInertialFrameOrientations=[[0, 0, 0, 1], [0, 0, 0, 1]],
            linkParentIndices=[0, 1],
            linkJointTypes=[p.JOINT_REVOLUTE, p.JOINT_REVOLUTE],
            linkJointAxis=[[0, 0, 1], [0, 0, 1]],
        )

        for j in [0, 1]:
            p.changeDynamics(robot, j, linearDamping=0.0, angularDamping=float(damping))

        return robot, 1

    def _inverse_kinematics_2link(self, x: float, y: float, l1: float, l2: float) -> tuple[float, float]:
        r2 = x * x + y * y
        c2 = (r2 - l1 * l1 - l2 * l2) / (2.0 * l1 * l2)
        c2 = float(np.clip(c2, -1.0, 1.0))
        s2 = float(np.sqrt(max(0.0, 1.0 - c2 * c2)))
        q2 = float(np.arctan2(s2, c2))
        q1 = float(np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2))
        return q1, q2

    def _forward_kinematics_2link(self, q1: float, q2: float, l1: float, l2: float) -> tuple[float, float]:
        x = l1 * np.cos(q1) + l2 * np.cos(q1 + q2)
        y = l1 * np.sin(q1) + l2 * np.sin(q1 + q2)
        return float(x), float(y)

    def _rollout(self, design: np.ndarray, cfg: dict, return_trace: bool = False):
        l1, l2, motor_strength, kp, kd, damping = [float(v) for v in design]

        cid = p.connect(p.DIRECT)
        try:
            robot, _ = self._build_robot(l1, l2, cfg["payload_kg"], damping)
            q1_t, q2_t = self._inverse_kinematics_2link(cfg["target_x"], cfg["target_y"], l1, l2)

            err_trace = []
            tau_trace = []
            ee_trace = []
            energy = 0.0

            for _step in range(int(cfg["sim_steps"])):
                for j, q_t in enumerate([q1_t, q2_t]):
                    p.setJointMotorControl2(
                        bodyUniqueId=robot,
                        jointIndex=j,
                        controlMode=p.POSITION_CONTROL,
                        targetPosition=q_t,
                        positionGain=float(kp) / 120.0,
                        velocityGain=float(kd) / 50.0,
                        force=float(cfg["torque_limit"]) * float(motor_strength),
                    )

                if cfg["disturbance_scale"] > 0:
                    disturb = self.np_random.normal(0.0, cfg["disturbance_scale"], size=2)
                    p.applyExternalTorque(robot, 0, [0, 0, float(disturb[0])], p.LINK_FRAME)
                    p.applyExternalTorque(robot, 1, [0, 0, float(disturb[1])], p.LINK_FRAME)

                p.stepSimulation()

                js0 = p.getJointState(robot, 0)
                js1 = p.getJointState(robot, 1)
                q1, q2 = float(js0[0]), float(js1[0])
                dq1, dq2 = float(js0[1]), float(js1[1])
                tau1, tau2 = float(js0[3]), float(js1[3])

                ee_x, ee_y = self._forward_kinematics_2link(q1, q2, l1, l2)
                err = float(np.sqrt((ee_x - cfg["target_x"]) ** 2 + (ee_y - cfg["target_y"]) ** 2))

                err_trace.append(err)
                tau_trace.append((tau1, tau2))
                ee_trace.append((ee_x, ee_y))
                energy += (abs(tau1 * dq1) + abs(tau2 * dq2)) * float(cfg["dt"])

            final_error = float(err_trace[-1])
            obj = np.array([final_error, float(energy)], dtype=np.float32)

            if return_trace:
                trace = {
                    "ee_trace": np.array(ee_trace, dtype=np.float32),
                    "err_trace": np.array(err_trace, dtype=np.float32),
                    "tau_trace": np.array(tau_trace, dtype=np.float32),
                    "target": np.array([cfg["target_x"], cfg["target_y"]], dtype=np.float32),
                    "design": np.array(design, dtype=np.float32),
                    "objectives": obj,
                }
                return obj, trace

            return obj
        finally:
            p.disconnect(cid)

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(design.astype(np.float32), self.design_space.low, self.design_space.high)
        return self._rollout(x, cfg, return_trace=False)

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        cfg = {**self.config.__dict__, **(config or {})}
        x = np.clip(starting_point.astype(np.float32), self.design_space.low, self.design_space.high)

        best = x.copy()
        best_obj = self.simulate(best, cfg)
        best_score = float(best_obj[0] + 0.02 * best_obj[1])

        history = [OptiStep(obj_values=best_obj, step=0)]
        step_scale = np.array([0.05, 0.05, 2.5, 8.0, 1.2, 0.08], dtype=np.float32)

        for step in range(1, int(cfg["max_iter"]) + 1):
            candidate = best + self.np_random.normal(0.0, 1.0, size=6).astype(np.float32) * step_scale
            candidate = np.clip(candidate, self.design_space.low, self.design_space.high)

            if self.check_constraints(candidate, cfg):
                history.append(OptiStep(obj_values=np.array([np.inf, np.inf], dtype=np.float32), step=step))
                continue

            obj = self.simulate(candidate, cfg)
            score = float(obj[0] + 0.02 * obj[1])
            if score < best_score:
                best, best_obj, best_score = candidate, obj, score

            history.append(OptiStep(obj_values=best_obj, step=step))

        return best, history

    def render(self, design: np.ndarray, *, open_window: bool = False):
        import matplotlib.pyplot as plt

        cfg = self.config.__dict__
        x = np.clip(design.astype(np.float32), self.design_space.low, self.design_space.high)
        obj, trace = self._rollout(x, cfg, return_trace=True)

        ee = trace["ee_trace"]
        err = trace["err_trace"]
        target = trace["target"]
        tau = trace["tau_trace"]

        fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))

        labels = ["link1", "link2", "motor", "kp", "kd", "damping"]
        axes[0].bar(labels, x, color=['#4c78a8', '#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2'])
        axes[0].set_title("Design variables")
        axes[0].tick_params(axis='x', rotation=35)

        axes[1].plot(ee[:, 0], ee[:, 1], lw=2, label="end-effector path")
        axes[1].scatter([target[0]], [target[1]], c='red', marker='x', s=70, label='target')
        r = x[0] + x[1]
        circle = plt.Circle((0, 0), r, color='gray', fill=False, linestyle='--', alpha=0.5)
        axes[1].add_patch(circle)
        axes[1].set_aspect('equal', 'box')
        axes[1].set_title("Task-space trajectory")
        axes[1].set_xlabel('x [m]')
        axes[1].set_ylabel('y [m]')
        axes[1].legend(fontsize=8)

        axes[2].plot(err, color='#e45756')
        axes[2].set_title("Tracking error over time")
        axes[2].set_xlabel("step")
        axes[2].set_ylabel("error [m]")
        axes[2].grid(alpha=0.3)

        axes[3].plot(np.abs(tau[:, 0]), label='|tau1|')
        axes[3].plot(np.abs(tau[:, 1]), label='|tau2|')
        axes[3].set_title("Actuation effort")
        axes[3].set_xlabel("step")
        axes[3].set_ylabel("torque [Nm]")
        axes[3].legend(fontsize=8)
        axes[3].grid(alpha=0.3)

        fig.suptitle(
            f"Objectives: final_error={obj[0]:.4f} m, energy={obj[1]:.3f} J",
            y=1.03,
        )
        fig.tight_layout()

        if open_window:
            plt.show()
        return fig, axes

    def random_design(self):
        d = self.np_random.uniform(self.design_space.low, self.design_space.high).astype(np.float32)
        return d, -1


### Step 3 - Smoke-test the scaffold

Validate behavior with simple checks before scaling to real domains.


Use the multi-panel render to read **where heat enters**, **how material is distributed**, and **where thermal bottlenecks remain**.


Use the final figure to interpret whether the design/controller combination reaches the target robustly with acceptable energy use.


In [ ]:
problem = PlanarManipulatorCoDesignProblem(
    seed=42,
    target_x=0.9,
    target_y=0.45,
    payload_kg=0.8,
    disturbance_scale=0.04,
    sim_steps=220,
    max_iter=40,
)
start, _ = problem.random_design()

cfg = {
    'target_x': 0.9,
    'target_y': 0.45,
    'payload_kg': 0.8,
    'disturbance_scale': 0.04,
    'sim_steps': 220,
    'dt': 1.0 / 120.0,
    'torque_limit': 12.0,
    'max_iter': 40,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [tracking_error_m, energy_J]:', obj0.tolist())
print('final objectives   [tracking_error_m, energy_J]:', objf.tolist())
print('optimization steps:', len(history))
print('How to read plots: vars | task-space path | error timeline | torque timeline')

problem.render(opt_design)


## Mapping to real EngiBench contributions

Use this template to onboard new domains while preserving common evaluation semantics.


## Contribution checklist

Check for leakage risks, undocumented defaults, and missing reproducibility metadata before contribution.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?


## Optional extension - Build dataset and train an EngiOpt generative model

This extension runs a full offline loop using an **existing EngiOpt model** (`cgan_1d`) on top of your custom simulator problem:

1. Generate a feasible dataset from simulator rollouts.
2. Keep a top-performing subset.
3. Train EngiOpt `cgan_1d` (`Generator` + `Discriminator`) for conditional generation.
4. Compare generated designs vs a random-design baseline.

Why this is useful:
- It demonstrates reuse of existing model infrastructure from EngiOpt.
- It clarifies how to adapt EngiOpt model classes to new, custom problem scaffolds.


In [ ]:
# Optional extension controls (safe defaults)
from pathlib import Path
import sys

import numpy as np
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

RUN_OPTIONAL_SECTION = False  # Set True to run this optional extension
N_FEASIBLE_SAMPLES = 260
TOP_FRACTION = 0.35
EPOCHS = 30
BATCH_SIZE = 64
LATENT_DIM = 8
FAST_SIM_CFG = {'sim_steps': 80, 'dt': 1.0 / 120.0}
EVAL_SAMPLES = 40

if 'problem' not in globals():
    problem = PlanarManipulatorCoDesignProblem(seed=7)

if 'google.colab' in sys.modules:
    OPTIONAL_ARTIFACT_DIR = Path('/content/dcc26_optional_artifacts')
else:
    OPTIONAL_ARTIFACT_DIR = Path('workshops/dcc26/optional_artifacts')
OPTIONAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Optional artifacts dir: {OPTIONAL_ARTIFACT_DIR.resolve()}')
print('Optional section enabled:' if RUN_OPTIONAL_SECTION else 'Optional section disabled:', RUN_OPTIONAL_SECTION)


In [ ]:
# Build offline dataset from simulator rollouts
rng = np.random.default_rng(123)


def sample_condition_dict() -> dict:
    return {
        'target_x': float(rng.uniform(0.20, 1.35)),
        'target_y': float(rng.uniform(0.05, 1.20)),
        'payload_kg': float(rng.uniform(0.0, 2.0)),
        'disturbance_scale': float(rng.uniform(0.0, 0.30)),
    }


def cond_to_vec(cfg: dict) -> np.ndarray:
    return np.array([
        cfg['target_x'],
        cfg['target_y'],
        cfg['payload_kg'],
        cfg['disturbance_scale'],
    ], dtype=np.float32)


def objective_score(obj: np.ndarray) -> float:
    return float(obj[0] + 0.02 * obj[1])


def make_dataset(problem_obj, n_feasible: int):
    designs, conds, objs = [], [], []
    max_attempts = n_feasible * 8
    attempts = 0

    while len(designs) < n_feasible and attempts < max_attempts:
        attempts += 1
        d, _ = problem_obj.random_design()
        cfg = sample_condition_dict()

        if len(problem_obj.check_constraints(d, cfg)) > 0:
            continue

        obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
        designs.append(d.astype(np.float32))
        conds.append(cond_to_vec(cfg))
        objs.append(obj.astype(np.float32))

        if len(designs) % 40 == 0:
            print(f'Collected feasible samples: {len(designs)}/{n_feasible}')

    if len(designs) < max(48, n_feasible // 3):
        raise RuntimeError(f'Not enough feasible samples ({len(designs)}).')

    designs = np.stack(designs)
    conds = np.stack(conds)
    objs = np.stack(objs)
    scores = np.array([objective_score(o) for o in objs], dtype=np.float32)

    keep_n = max(48, int(TOP_FRACTION * len(scores)))
    top_idx = np.argsort(scores)[:keep_n]

    data = {
        'designs_all': designs,
        'conditions_all': conds,
        'objectives_all': objs,
        'scores_all': scores,
        'designs_top': designs[top_idx],
        'conditions_top': conds[top_idx],
        'objectives_top': objs[top_idx],
        'scores_top': scores[top_idx],
    }
    return data


if RUN_OPTIONAL_SECTION:
    dataset = make_dataset(problem, N_FEASIBLE_SAMPLES)
    np.savez(OPTIONAL_ARTIFACT_DIR / 'manipulator_dataset.npz', **dataset)
    print('Saved dataset:', OPTIONAL_ARTIFACT_DIR / 'manipulator_dataset.npz')
    print('All samples:', dataset['designs_all'].shape[0], '| Top samples:', dataset['designs_top'].shape[0])
else:
    dataset = None
    print('Skipped dataset creation. Set RUN_OPTIONAL_SECTION=True to run this block.')


### Optional model - EngiOpt `cgan_1d`

This cell reuses `engiopt.cgan_1d.cgan_1d` classes directly:
- `Normalizer`
- `Generator`
- `Discriminator`

Adapter note:
- `Discriminator` in this module expects module-level `design_shape` and `n_conds` symbols.
- We set those explicitly before instantiation to keep behavior aligned with the original script.


In [ ]:
# Train EngiOpt cgan_1d on the top-performing subset
if RUN_OPTIONAL_SECTION:
    import engiopt.cgan_1d.cgan_1d as cgan1d

    device = th.device('cuda' if th.cuda.is_available() else 'cpu')

    x_cond = dataset['conditions_top'].astype(np.float32)
    y_design = dataset['designs_top'].astype(np.float32)

    cond_t = th.tensor(x_cond, dtype=th.float32, device=device)
    design_t = th.tensor(y_design, dtype=th.float32, device=device)

    cond_min = cond_t.amin(dim=0)
    cond_max = cond_t.amax(dim=0)
    design_min = design_t.amin(dim=0)
    design_max = design_t.amax(dim=0)

    conds_normalizer = cgan1d.Normalizer(cond_min, cond_max)
    design_normalizer = cgan1d.Normalizer(design_min, design_max)

    design_shape = (design_t.shape[1],)
    n_conds = cond_t.shape[1]

    # Compatibility shim for cgan_1d.Discriminator internal references.
    cgan1d.design_shape = design_shape
    cgan1d.n_conds = n_conds

    generator = cgan1d.Generator(
        latent_dim=LATENT_DIM,
        n_conds=n_conds,
        design_shape=design_shape,
        design_normalizer=design_normalizer,
        conds_normalizer=conds_normalizer,
    ).to(device)

    discriminator = cgan1d.Discriminator(
        conds_normalizer=conds_normalizer,
        design_normalizer=design_normalizer,
    ).to(device)

    loader = DataLoader(TensorDataset(design_t, cond_t), batch_size=BATCH_SIZE, shuffle=True)

    adv_loss = nn.BCELoss()
    opt_g = th.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = th.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

    g_hist, d_hist = [], []

    for epoch in range(1, EPOCHS + 1):
        g_epoch, d_epoch, n_steps = 0.0, 0.0, 0
        for real_design, cond in loader:
            bs = real_design.shape[0]
            valid = th.ones((bs, 1), device=device)
            fake = th.zeros((bs, 1), device=device)

            # Generator update
            opt_g.zero_grad()
            z = th.randn((bs, LATENT_DIM), device=device)
            gen_design = generator(z, cond)
            g_loss = adv_loss(discriminator(gen_design, cond), valid)
            g_loss.backward()
            opt_g.step()

            # Discriminator update
            opt_d.zero_grad()
            real_loss = adv_loss(discriminator(real_design, cond), valid)
            fake_loss = adv_loss(discriminator(gen_design.detach(), cond), fake)
            d_loss = 0.5 * (real_loss + fake_loss)
            d_loss.backward()
            opt_d.step()

            g_epoch += float(g_loss.item())
            d_epoch += float(d_loss.item())
            n_steps += 1

        g_hist.append(g_epoch / max(1, n_steps))
        d_hist.append(d_epoch / max(1, n_steps))
        if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
            print(f'Epoch {epoch:02d}/{EPOCHS} | g_loss={g_hist[-1]:.6f} | d_loss={d_hist[-1]:.6f}')

    th.save(
        {
            'generator': generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'cond_min': cond_min.cpu(),
            'cond_max': cond_max.cpu(),
            'design_min': design_min.cpu(),
            'design_max': design_max.cpu(),
        },
        OPTIONAL_ARTIFACT_DIR / 'engiopt_cgan1d_weights.pt',
    )
    print('Saved model:', OPTIONAL_ARTIFACT_DIR / 'engiopt_cgan1d_weights.pt')
else:
    generator, discriminator = None, None
    g_hist, d_hist = [], []
    device = th.device('cpu')
    print('Skipped model training. Set RUN_OPTIONAL_SECTION=True to run this block.')


In [ ]:
# Evaluate generated designs vs random baseline
import matplotlib.pyplot as plt


def sample_baseline(problem_obj, cfg: dict, trials: int = 8):
    best_obj = None
    for _ in range(trials):
        d, _ = problem_obj.random_design()
        if len(problem_obj.check_constraints(d, cfg)) > 0:
            continue
        obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
        if best_obj is None or objective_score(obj) < objective_score(best_obj):
            best_obj = obj
    if best_obj is None:
        d, _ = problem_obj.random_design()
        best_obj = problem_obj.simulate(d, {**cfg, **FAST_SIM_CFG})
    return best_obj


def generate_design(generator_obj, cfg_vec: np.ndarray, lb: np.ndarray, ub: np.ndarray):
    # cgan_1d Generator uses BatchNorm; switch to eval for single-sample inference.
    was_training = generator_obj.training
    generator_obj.eval()
    try:
        with th.no_grad():
            c = th.tensor(cfg_vec[None, :], dtype=th.float32, device=device)
            z = th.randn((1, LATENT_DIM), dtype=th.float32, device=device)
            d = generator_obj(z, c).cpu().numpy()[0]
    finally:
        if was_training:
            generator_obj.train()
    return np.clip(d.astype(np.float32), lb, ub)


if RUN_OPTIONAL_SECTION:
    lb = problem.design_space.low.astype(np.float32)
    ub = problem.design_space.high.astype(np.float32)

    gen_objs = []
    base_objs = []
    feasible_count = 0

    for _ in range(EVAL_SAMPLES):
        cfg = sample_condition_dict()
        cfg_vec = cond_to_vec(cfg)

        gen_obj = None
        for _retry in range(6):
            d_gen = generate_design(generator, cfg_vec, lb, ub)
            if len(problem.check_constraints(d_gen, cfg)) == 0:
                gen_obj = problem.simulate(d_gen, {**cfg, **FAST_SIM_CFG})
                feasible_count += 1
                break
        if gen_obj is None:
            d_fallback, _ = problem.random_design()
            gen_obj = problem.simulate(d_fallback, {**cfg, **FAST_SIM_CFG})

        base_obj = sample_baseline(problem, cfg, trials=8)
        gen_objs.append(gen_obj)
        base_objs.append(base_obj)

    gen_objs = np.stack(gen_objs)
    base_objs = np.stack(base_objs)

    summary = {
        'generated_error_mean': float(np.mean(gen_objs[:, 0])),
        'generated_energy_mean': float(np.mean(gen_objs[:, 1])),
        'baseline_error_mean': float(np.mean(base_objs[:, 0])),
        'baseline_energy_mean': float(np.mean(base_objs[:, 1])),
        'generated_feasible_rate': float(feasible_count / EVAL_SAMPLES),
    }

    print('Optional extension summary (EngiOpt cgan_1d):')
    for k, v in summary.items():
        print(f'  {k}: {v:.6f}')

    np.savez(
        OPTIONAL_ARTIFACT_DIR / 'optional_eval_summary_engiopt_cgan1d.npz',
        gen_objs=gen_objs,
        base_objs=base_objs,
        g_hist=np.array(g_hist, dtype=np.float32),
        d_hist=np.array(d_hist, dtype=np.float32),
        **summary,
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].plot(g_hist, label='g_loss')
    axes[0].plot(d_hist, label='d_loss')
    axes[0].set_title('EngiOpt cgan_1d training losses')
    axes[0].set_xlabel('epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].hist(base_objs[:, 0], bins=12, alpha=0.6, label='baseline')
    axes[1].hist(gen_objs[:, 0], bins=12, alpha=0.6, label='generated')
    axes[1].set_title('Final tracking error')
    axes[1].set_xlabel('error [m]')
    axes[1].legend()

    axes[2].hist(base_objs[:, 1], bins=12, alpha=0.6, label='baseline')
    axes[2].hist(gen_objs[:, 1], bins=12, alpha=0.6, label='generated')
    axes[2].set_title('Actuation energy')
    axes[2].set_xlabel('energy [J]')
    axes[2].legend()

    fig.tight_layout()
    plt.show()
else:
    print('Skipped evaluation. Set RUN_OPTIONAL_SECTION=True to run this block.')


### Discussion prompts for workshop synthesis

1. How portable are EngiOpt models across domains with different design representations?
2. What is the minimum adapter contract needed to reuse a model on a new problem?
3. Should benchmark reporting require both feasibility and objective trade-off metrics for generated designs?
